# Comprobación de Supuestos en Experimentos

---

### ¿Por qué comprobar supuestos?

- ### Antes de aplicar una prueba estadística (como la prueba t o ANOVA), debemos verificar que los datos cumplan con los supuestos requeridos. 
- ### Si los supuestos se violan, los resultados de la prueba pueden ser inválidos (aumentando el riesgo de cometer errores Tipo I o Tipo II).

### Objetivos

- ### 1. **Normalidad:** Evaluar si los datos (o los residuos) siguen una distribución normal usando un **QQ Plot** y la prueba de **Shapiro-Wilk**.
- ### 2. **Homocedasticidad:** Evaluar si las varianzas de los grupos son iguales usando la prueba de **Levene**.
- ### 3. **Interpretar** los resultados y decidir qué prueba estadística usar.

### 1. Importar las Librerías Necesarias

### Usaremos las siguientes librerías:
- ### **`numpy`**: Para generar y manipular datos.
- ### **`scipy.stats`**: Para las pruebas estadísticas (Shapiro-Wilk, Levene).
- ### **`matplotlib`** y **`seaborn`**: Para el QQ Plot y visualizaciones.
- ### **`statsmodels`**: Para el QQ Plot y modelos estadísticos.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

print("Librerías importadas correctamente.")

### 2. Crear Datos de Ejemplo

### Simular un experimento en Ingeniería de Software donde se midió el **tiempo de compilación** (en segundos) de un proyecto usando **tres herramientas diferentes** (A, B y C).

- ### **Herramienta A:** Tiempos con distribución normal.
- ### **Herramienta B:** Tiempos con distribución normal pero con mayor varianza.
- ### **Herramienta C:** Tiempos con distribución sesgada (no normal).

In [ ]:
# ============================================================
# SIMULACIÓN DE DATOS PARA UN EXPERIMENTO EN INGENIERÍA DE SOFTWARE
# ============================================================
# Objetivo: Crear tres grupos de datos que simulan los tiempos
# de compilación de tres herramientas diferentes para usarlos
# en un análisis de varianza (ANOVA) o pruebas de supuestos.
# ============================================================

# ---- Paso 1: Fijar la semilla de aleatoriedad ----
# np.random.seed(42) asegura que los números aleatorios generados
# sean SIEMPRE los mismos. Esto es crucial para la reproducibilidad
# de los experimentos. Si no se fija, cada ejecución daría resultados
# distintos.
np.random.seed(42)

# ---- Paso 2: Simular los datos de la Herramienta A ----
# Usamos np.random.normal() para generar datos con distribución normal.
# Parámetros:
#   - loc=50: La media del tiempo de compilación es 50 ms/segundos.
#   - scale=5: La desviación estándar es 5 (varianza baja/moderada).
#   - size=30: Se generan 30 observaciones (tamaño de muestra).
tiempos_A = np.random.normal(loc=50, scale=5, size=30)

#   - scale=12: La desviación estándar es 12 (varianza alta, datos más dispersos).
tiempos_B = np.random.normal(loc=52, scale=12, size=30)

#   - scale=10: La media de la distribución exponencial es 10.
#   - size=30: 30 observaciones.
# Luego se suma +40 para desplazar los datos a un rango similar
# al de las otras herramientas (para que tengan sentido en el contexto).
tiempos_C = np.random.exponential(scale=10, size=30) + 40

# ---- Paso 3: Crear un DataFrame de Pandas ----
# columnas:
#   - 'herramienta': Indica a qué grupo pertenece (A, B o C).
#   - 'tiempo': El valor numérico del tiempo de compilación.
df = pd.DataFrame({
    # Columna 'herramienta': Creamos una lista con 30 'A', 30 'B' y 30 'C'.
    # Esto asigna la etiqueta correcta a cada grupo.
    'herramienta': ['A'] * 30 + ['B'] * 30 + ['C'] * 30,
    
    # Columna 'tiempo': Concatenamos los tres arrays de tiempos
    # en un solo array largo (90 valores en total).
    'tiempo': np.concatenate([tiempos_A, tiempos_B, tiempos_C])
})

# ---- Paso 6: Mostrar las primeras 10 filas ----
# df.head(10) imprime las primeras 10 filas del DataFrame
# para verificar que los datos se crearon correctamente.
print(df.head(10))

# 3. Comprobación de Normalidad: QQ Plot

### El **QQ Plot** compara los cuantiles observados de los datos con los cuantiles esperados de una distribución normal.
- ### Si los puntos se alinean aproximadamente en una línea recta, los datos son compatibles con la normalidad.
- ### Si hay curvatura, los datos se desvían de la normalidad.

In [ ]:
# ============================================================
# CREACIÓN DE TRES QQ PLOTS COMPARATIVOS
# ============================================================
# Objetivo: Generar un gráfico con tres paneles (uno al lado del
# otro) para comparar visualmente la normalidad de los datos de
# tres herramientas diferentes (A, B y C).
# ============================================================

# ---- Paso 1: Crear la figura y los ejes ----
# plt.subplots(1, 3) crea una cuadrícula de gráficos:
#   - 1: número de filas (un solo nivel vertical).
#   - 3: número de columnas (tres gráficos lado a lado).
# figsize=(15, 5) define el tamaño total de la figura en pulgadas
# 'fig' es el lienzo completo y 'ax' es un arreglo con los 3 ejes.
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

# ---- Paso 2: QQ Plot para la Herramienta A ----
# sm.qqplot() genera el gráfico de cuantiles-cuantiles.
# Parámetros:
#   - tiempos_A: Los datos de la Herramienta A.
#   - line='s': Dibuja una línea de referencia diagonal.
#               's' significa "standardized" (estandarizada).
#   - ax=ax[0]: Indica que el gráfico se dibuje en el primer panel.
sm.qqplot(tiempos_A, line='s', ax=ax[0])
# Agrega un título al primer panel.
ax[0].set_title('Herramienta A (Normal)')

# ---- Paso 3: QQ Plot para la Herramienta B ----
# Repetimos el proceso, pero usando el segundo panel (ax[1]).
# Esta herramienta también es normal, pero tiene mayor varianza.
sm.qqplot(tiempos_B, line='s', ax=ax[1])
ax[1].set_title('Herramienta B (Normal, varianza alta)')

# ---- Paso 4: QQ Plot para la Herramienta C ----
# Repetimos el proceso para el tercer panel (ax[2]).
# Esta herramienta tiene una distribución exponencial (sesgada),
# por lo que se espera que los puntos NO sigan la línea recta.
sm.qqplot(tiempos_C, line='s', ax=ax[2])
ax[2].set_title('Herramienta C (Sesgada)')

# ---- Paso 5: Ajustar el diseño ----
# plt.tight_layout() ajusta automáticamente los márgenes y el
# espaciado entre los tres gráficos para que los títulos y las
# etiquetas no se superpongan.
plt.tight_layout()

# ---- Paso 6: Mostrar la figura ----
# plt.show() renderiza y muestra los tres gráficos en pantalla.
plt.show()

# 4. Comprobación de Normalidad: Shapiro-Wilk

### La **prueba de Shapiro-Wilk** evalúa formalmente si los datos son compatibles con una distribución normal.
- ### **$H_0$:** Los datos siguen una distribución normal.
- ### **$H_1$:** Los datos no siguen una distribución normal.
- ### Si $p < 0.05$, rechazamos $H_0$ (los datos **no** son normales).
- ### Si $p \geq 0.05$, no rechazamos $H_0$ (los datos **parecen** ser normales).

In [ ]:
# ============================================================
# PRUEBA DE NORMALIDAD DE SHAPIRO-WILK PARA TRES HERRAMIENTAS
# ============================================================
# Objetivo: Evaluar si los tiempos de compilación de cada
# herramienta (A, B y C) siguen una distribución normal.
# Esta prueba es fundamental antes de aplicar ANOVA o prueba t.
# ============================================================

print("PRUEBA DE SHAPIRO-WILK")

# ---- Paso 1: Recorrer cada herramienta ----
# La lista contiene tuplas: (nombre_del_grupo, array_de_datos).
for nombre, datos in [('A', tiempos_A), ('B', tiempos_B), ('C', tiempos_C)]:
    
    # ---- Paso 2: Ejecutar la prueba de Shapiro-Wilk ----
    # stats.shapiro() devuelve dos valores:
    #   - stat: El estadístico W (un número entre 0 y 1).
    #   - p_valor: La probabilidad de observar estos datos si H0 fuera cierta.
    # H0: Los datos siguen una distribución normal.
    # H1: Los datos NO siguen una distribución normal.
    stat, p_valor = stats.shapiro(datos)
    
    # ---- Paso 3: Imprimir los resultados de la prueba ----
    print(f"\nHerramienta {nombre}:")
    print(f"  Estadístico W: {stat:.4f}")
    print(f"  Valor p: {p_valor:.4f}")
    
    # ---- Paso 4: Tomar la decisión ----
    # Comparamos el valor p con el nivel de significancia (alpha = 0.05).    
    if p_valor < 0.05:
        # Es muy poco probable que los  datos provengan de una normal, así que se rechaza la idea de que sean normales"
        # Si p < 0.05, rechazamos H0. Hay evidencia de que los datos NO son normales.
        print(f"  Decisión: Rechazar H0 → Los datos NO son normales.")
    else:
        # Si p >= 0.05, no rechazamos H0. No hay evidencia suficiente para decir que no son normales.
        print(f"  Decisión: No rechazar H0 → Los datos parecen ser normales.")

# 5. Comprobación de Homocedasticidad: Levene

### La **prueba de Levene** evalúa si las varianzas de los grupos son aproximadamente iguales (de acuerdo al nivel de significancia $\alpha$ = 0.5).
- ### **$H_0$:** Las varianzas son iguales ($\sigma_A^2 = \sigma_B^2 = \sigma_C^2$).
- ### **$H_1$:** Al menos una varianza es diferente.
- ### Si $p < 0.05$, rechazamos $H_0$ (las varianzas **no** son iguales).
- ### Si $p \geq 0.05$, no rechazamos $H_0$ (las varianzas **parecen** ser iguales).

In [ ]:
# Prueba de Levene
stat_levene, p_levene = stats.levene(tiempos_A, tiempos_B, tiempos_C)

print("=" * 50)
print("PRUEBA DE LEVENE (Homocedasticidad)")
print("=" * 50)
print(f"Estadístico W: {stat_levene:.4f}")
print(f"Valor p: {p_levene:.4f}")
if p_levene < 0.05:
    print("Decisión: Rechazar H0 → Las varianzas SON diferentes.")
    print("Recomendación: Usar la prueba t de Welch o ANOVA de Welch.")
else:
    print("Decisión: No rechazar H0 → Las varianzas parecen ser iguales.")
    print("Recomendación: Puedes usar la prueba t de Student o ANOVA estándar.")

# 6. Visualización Comparativa: Boxplot y Varianzas

### Para complementar las pruebas, visualicemos la dispersión de cada grupo.

In [ ]:
# Boxplot comparativo
plt.figure(figsize=(10, 6))
sns.boxplot(x='herramienta', y='tiempo', data=df, hue='herramienta', palette='Set2', legend=False)
plt.title('Comparación de tiempos de compilación por herramienta')
plt.xlabel('Herramienta')
plt.ylabel('Tiempo (segundos)')
plt.show()

# Calcular varianzas
print("Varianzas por herramienta:")
print(df.groupby('herramienta')['tiempo'].var())

# Ejercicio 1: Comprobación de Supuestos en un Experimento de QA

###  Contexto
### Un equipo de QA está evaluando dos técnicas de pruebas diferentes (A y B) para detectar defectos en módulos de software. Se midió el **tiempo de detección del primer defecto** (en minutos) en 25 módulos independientes para cada técnica.

### Los datos son los siguientes:

```python
datos_A = [
    9.828739, 13.994691, 12.565957, 8.987411, 10.842799,
    15.302873, 7.146642, 11.142175, 14.531873, 10.266519,
    10.642228, 11.810582, 14.982779, 10.722196, 11.112036,
    11.131297, 16.411860, 16.373572, 14.008108, 12.772373,
    13.474737, 14.981464, 10.128332, 14.351658, 9.492239
]

datos_B = [
    8.811242, 16.535526, 4.856596, 11.299656, 7.691226,
    10.721903, 1.992946, 3.142334, 8.500614, 16.637312,
    11.131822, 12.014230, 15.441114, 7.602318, 13.418137,
    7.973167, 3.361653, 10.045501, 14.869029, 13.692945,
    11.940848, 23.961826, 14.064561, 16.893680, 23.190717
]
```

### Instrucciones y Preguntas

- ### 1. Normalidad (Visual): Genera un QQ Plot para cada técnica. Observa si los puntos se alinean con la línea recta y si hay curvaturas o valores atípicos que sugieran desviaciones de la normalidad.
- ### 2. Normalidad (Formal): Aplica la prueba de Shapiro-Wilk a cada técnica. Determina si los datos son normales justificando tu respuesta con el valor $p$ obtenido.
- ### 3. Homocedasticidad: Aplica la prueba de Levene para comparar las varianzas de ambas técnicas. Indica si se cumple el supuesto de varianzas iguales.
- ### 4. Decisión: Basándote en los resultados anteriores, ¿qué prueba estadística usarías para comparar las medias de ambas técnicas? Justifica tu respuesta eligiendo entre la prueba $t$ de Student, la prueba $t$ de Welch


In [20]:
# TODO:

datos_A = [
    9.828739, 13.994691, 12.565957, 8.987411, 10.842799,
    15.302873, 7.146642, 11.142175, 14.531873, 10.266519,
    10.642228, 11.810582, 14.982779, 10.722196, 11.112036,
    11.131297, 16.411860, 16.373572, 14.008108, 12.772373,
    13.474737, 14.981464, 10.128332, 14.351658, 9.492239
]

datos_B = [
    8.811242, 16.535526, 4.856596, 11.299656, 7.691226,
    10.721903, -1.992946, 3.142334, 8.500614, 16.637312,
    11.131822, 12.014230, 15.441114, 7.602318, 13.418137,
    7.973167, 3.361653, 10.045501, 14.869029, 13.692945,
    11.940848, 23.961826, 14.064561, 16.893680, 23.190717
]





# Ejercicio 2 con datasets

# Cargar el Dataset Iris

### El **Iris dataset** es uno de los más clásicos en estadística y machine learning. Contiene mediciones de 150 flores de 3 especies diferentes:

- ### **Setosa**
- ### **Versicolor**
- ### **Virginica**

### Para cada flor se midieron 4 variables numéricas:

| **Variable** | **Descripción** | **Unidad** |
| :--- | :--- | :---: |
| `sepal length` | Longitud del sépalo | cm |
| `sepal width` | Ancho del sépalo | cm |
| `petal length` | Longitud del pétalo | cm |
| `petal width` | Ancho del pétalo | cm |
| `species` | Especie de la flor | — |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar el dataset Iris desde seaborn ( viene incluido)
df = sns.load_dataset('iris')

# Verificar la carga
print(f"Dimensiones: {df.shape}")
print()
print(df.head())

## Ejercicio 1: Comprobación de Supuestos con Iris

### Contexto
### Usa el dataset Iris para evaluar si se cumplen los supuestos de normalidad y homocedasticidad para la variable `petal_length` en las tres especies.

### Instrucciones
- ### 1. Aplica Shapiro-Wilk a `petal_length` para cada especie. ¿Son normales?
- ### 2. Aplica Levene a `petal_length` agrupado por especie. ¿Las varianzas son iguales?
- ### 3. Genera un QQ Plot para cada especie. ¿Se alinean los puntos?
- ### 4. Con base en los resultados, ¿qué prueba usarías para comparar las medias de `petal_length` entre especies? (ANOVA o ANOVA de Welch).


In [ ]:
# TODO: Ejercicio 1








## Ejercicio 2: Comprobación de Supuestos con Iris

### Contexto
### Usa el dataset Iris para evaluar si se cumplen los supuestos de normalidad y homocedasticidad para la variable `sepal_length` en las tres especies.

### Instrucciones
- ### 1. Aplica Shapiro-Wilk a `sepal_length` para cada especie. ¿Son normales?
- ### 2. Aplica Levene a `sepal_length` agrupado por especie. ¿Las varianzas son iguales?
- ### 3. Genera un QQ Plot para cada especie. ¿Se alinean los puntos?
- ### 4. Con base en los resultados, ¿qué prueba usarías para comparar las medias de `sepal_length` entre especies? (ANOVA o ANOVA de Welch).


In [25]:
# TODO: Ejercicio 2






